[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # Initialize projections
        assert num_heads % num_kv_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads

        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        # Self-attention with grouped KV
        # (batch, seq, d_model)
        batch, seq, _ = x.shape
        queries = self.W_q(x).view(batch, seq, self.num_heads, self.d_k).transpose(1, 2) # (batch, num_heads, seq, d_k)
        keys = self.W_k(x).view(batch, seq, self.num_kv_heads, self.d_k).transpose(1, 2) # (batch, num_kv_heads, seq, dk)
        values = self.W_v(x).view(batch, seq, self.num_kv_heads, self.d_k).transpose(1, 2) # (batch, num_kv_heads, seq, dk)

        # repeat interleave to expand the head dimension 
        keys = keys.repeat_interleave(self.num_heads // self.num_kv_heads, dim=1) # (batch, num_heads, seq, d_k)
        values = values.repeat_interleave(self.num_heads // self.num_kv_heads, dim=1) # (batch, num_heads, seq, d_k)

        # (batch, num_heads, seq, d_k) x (batch, num_heads, d_model, seq)
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.d_k) # (batch, num_heads, seq, seq)
        attn = torch.softmax(scores, dim=-1)

        # (batch, num_heads, seq, seq) x (batch, num_heads, seq, d_k)
        attn = (attn @ values).transpose(1, 2).contiguous().view(batch, seq, self.d_model)

        return self.W_o(attn)

        
        

In [16]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [17]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.6ms)
  ✅ [2/5] nn.Linear with correct shapes (0.5ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.5ms)
  ✅ [4/5] KV heads are shared correctly (1.8ms)
  ✅ [5/5] Gradient flow (4.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (10.9ms total)
  Progress saved. Run status() to see your dashboard.

